# PlantVillage Data Preprocessing (32x32 Grayscale)

Bu notebook, PlantVillage veri setindeki yaprak görüntülerini klasik
makine öğrenmesi yöntemleri (Logistic Regression, SVM, Random Forest vb.)
için hazırlar.

Adımlar:

1. Klasör yapısındaki sınıfları (class folder) okumak.
2. Görselleri 32x32 gri tonlamalı hale getirip [0, 1] aralığına normalize etmek.
3. Her sınıf için train / validation / test olacak şekilde veri bölmek.
4. Sınıf isimlerini (class_names.txt) ve işlenmiş veri setini
   `preprocessed_plantvillage/*.npy` dosyaları olarak kaydetmek.


# Cell 1 – Importlar ve ayarlar

In [1]:
import os
import random
from PIL import Image
import numpy as np

# -------------------
# Configuration
# -------------------

# This notebook is assumed to live under something like:
#   Project/
#       Dataset/
#           plantvillage/
#               color/
#       Soruce Code/
#           01_preprocess_plantvillage.ipynb  (this file)
#
# Adjust this path if your structure is different.
DATA_ROOT = "../Dataset/plantvillage/color"

OUTPUT_DIR = "preprocessed_plantvillage"

IMG_SIZE = 32  # target size (32x32)
RANDOM_SEED = 42

# Train / validation / test split ratios
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15  # test ratio = 1 - TRAIN_RATIO - VAL_RATIO

# Optional: limit max images per class for speed/debug (None = unlimited)
MAX_IMAGES_PER_CLASS = None

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)


DATA_ROOT → PlantVillage color klasörünün yolu.

Görselleri 32×32 gri yapacağız.

Her sınıftan veriyi train/val/test olarak böleceğiz.

Çıktılar preprocessed_plantvillage klasörüne kaydedilecek.

# Cell 2 – Yardımcı fonksiyonlar (sınıf isimleri + görüntü vektörü)

In [2]:
def get_class_names(root_dir):
    """
    Return sorted list of class folder names under root_dir.
    Each subfolder is treated as one class label.
    """
    class_names = []
    for item in os.listdir(root_dir):
        full_path = os.path.join(root_dir, item)
        if os.path.isdir(full_path):
            class_names.append(item)
    class_names = sorted(class_names)
    return class_names


def load_image_as_vector(path, img_size=IMG_SIZE):
    """
    Load an image file, convert it to grayscale, resize to img_size x img_size,
    normalize pixel values to [0, 1], and flatten to a 1D list (feature vector).
    """
    with Image.open(path) as img:
        img = img.convert("L")  # grayscale
        img = img.resize((img_size, img_size))
        pixels = list(img.getdata())  # length = img_size * img_size
        vector = [p / 255.0 for p in pixels]
        return vector


# Cell 3 – Dataset’i oluşturma (train/val/test split)

In [3]:
def build_splits_for_dataset(root_dir, train_ratio, val_ratio, max_per_class=None):
    """
    Walk through the dataset root directory which contains subfolders per class,
    split images into train / val / test splits for each class, and return
    X_train, y_train, X_val, y_val, X_test, y_test.

    Labels are integer indices 0..C-1, where C is the number of classes,
    based on sorted class folder names.
    """
    class_names = get_class_names(root_dir)
    print("Found classes:")
    for idx, name in enumerate(class_names):
        print(f"{idx:2d} -> {name}")
    print()

    X_train, y_train = [], []
    X_val, y_val = [], []
    X_test, y_test = [], []

    for class_idx, class_name in enumerate(class_names):
        class_dir = os.path.join(root_dir, class_name)
        image_files = [
            f for f in os.listdir(class_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ]

        if max_per_class is not None:
            random.shuffle(image_files)
            image_files = image_files[:max_per_class]
        else:
            random.shuffle(image_files)

        n_total = len(image_files)
        n_train = int(n_total * train_ratio)
        n_val = int(n_total * val_ratio)
        n_test = n_total - n_train - n_val

        train_files = image_files[:n_train]
        val_files = image_files[n_train:n_train + n_val]
        test_files = image_files[n_train + n_val:]

        print(f"Class '{class_name}' (idx {class_idx}): "
              f"total={n_total}, train={len(train_files)}, "
              f"val={len(val_files)}, test={len(test_files)}")

        # Process train images
        for fname in train_files:
            path = os.path.join(class_dir, fname)
            try:
                x_vec = load_image_as_vector(path)
            except Exception as e:
                print(f"Warning: cannot load {path}: {e}")
                continue
            X_train.append(x_vec)
            y_train.append(class_idx)

        # Process val images
        for fname in val_files:
            path = os.path.join(class_dir, fname)
            try:
                x_vec = load_image_as_vector(path)
            except Exception as e:
                print(f"Warning: cannot load {path}: {e}")
                continue
            X_val.append(x_vec)
            y_val.append(class_idx)

        # Process test images
        for fname in test_files:
            path = os.path.join(class_dir, fname)
            try:
                x_vec = load_image_as_vector(path)
            except Exception as e:
                print(f"Warning: cannot load {path}: {e}")
                continue
            X_test.append(x_vec)
            y_test.append(class_idx)

    print("\nTotal samples:")
    print("Train:", len(X_train))
    print("Val  :", len(X_val))
    print("Test :", len(X_test))

    return (
        np.array(X_train, dtype=np.float32),
        np.array(y_train, dtype=np.int64),
        np.array(X_val, dtype=np.float32),
        np.array(y_val, dtype=np.int64),
        np.array(X_test, dtype=np.float32),
        np.array(y_test, dtype=np.int64),
        class_names,
    )



# Cell 4 – Preprocess’i çalıştır

In [4]:
(
    X_train_pv,
    y_train_pv,
    X_val_pv,
    y_val_pv,
    X_test_pv,
    y_test_pv,
    class_names_pv,
) = build_splits_for_dataset(
    DATA_ROOT,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    max_per_class=MAX_IMAGES_PER_CLASS,
)

print("\nShapes:")
print("X_train:", X_train_pv.shape, "y_train:", y_train_pv.shape)
print("X_val  :", X_val_pv.shape, "y_val  :", y_val_pv.shape)
print("X_test :", X_test_pv.shape, "y_test :", y_test_pv.shape)



Found classes:
 0 -> Apple___Apple_scab
 1 -> Apple___Black_rot
 2 -> Apple___Cedar_apple_rust
 3 -> Apple___healthy
 4 -> Blueberry___healthy
 5 -> Cherry_(including_sour)___Powdery_mildew
 6 -> Cherry_(including_sour)___healthy
 7 -> Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
 8 -> Corn_(maize)___Common_rust_
 9 -> Corn_(maize)___Northern_Leaf_Blight
10 -> Corn_(maize)___healthy
11 -> Grape___Black_rot
12 -> Grape___Esca_(Black_Measles)
13 -> Grape___Leaf_blight_(Isariopsis_Leaf_Spot)
14 -> Grape___healthy
15 -> Orange___Haunglongbing_(Citrus_greening)
16 -> Peach___Bacterial_spot
17 -> Peach___healthy
18 -> Pepper,_bell___Bacterial_spot
19 -> Pepper,_bell___healthy
20 -> Potato___Early_blight
21 -> Potato___Late_blight
22 -> Potato___healthy
23 -> Raspberry___healthy
24 -> Soybean___healthy
25 -> Squash___Powdery_mildew
26 -> Strawberry___Leaf_scorch
27 -> Strawberry___healthy
28 -> Tomato___Bacterial_spot
29 -> Tomato___Early_blight
30 -> Tomato___Late_blight
31 -> Tomato__

# Cell 5 – .npy ve sınıf isimlerini kaydet

In [5]:
# Save numpy arrays
np.save(os.path.join(OUTPUT_DIR, "train_X.npy"), X_train_pv)
np.save(os.path.join(OUTPUT_DIR, "train_y.npy"), y_train_pv)
np.save(os.path.join(OUTPUT_DIR, "val_X.npy"), X_val_pv)
np.save(os.path.join(OUTPUT_DIR, "val_y.npy"), y_val_pv)
np.save(os.path.join(OUTPUT_DIR, "test_X.npy"), X_test_pv)
np.save(os.path.join(OUTPUT_DIR, "test_y.npy"), y_test_pv)

# Save class names to a text file (one name per line)
class_names_path = os.path.join(OUTPUT_DIR, "class_names.txt")
with open(class_names_path, "w", encoding="utf-8") as f:
    for name in class_names_pv:
        f.write(name + "\n")

print(f"\nSaved preprocessed PlantVillage data to '{OUTPUT_DIR}'")
print(f"Class names written to {class_names_path}")




Saved preprocessed PlantVillage data to 'preprocessed_plantvillage'
Class names written to preprocessed_plantvillage\class_names.txt


Sonuç:

preprocessed_plantvillage/train_X.npy, train_y.npy, val_*, test_*

preprocessed_plantvillage/class_names.txt